In [0]:
%sql 
-- PASO 1 : CREAR CATALOGO

CREATE CATALOG IF NOT EXISTS catalogo_ventas;

USE CATALOG catalogo_ventas;
     
SHOW CATALOGS
     



In [0]:
%sql
-- PASO 2: CREAR ESQUEMA RAW

CREATE SCHEMA IF NOT EXISTS catalogo_ventas.raw
COMMENT 'Esquema para datos crudos sin procesar(Bronze Layer)';

SHOW SCHEMAS

In [0]:
%sql
-- PASO 3: CREAR EL VOLUME PARA ARCHIVOS (ventas_helados_bronze.csv)

CREATE VOLUME IF NOT EXISTS catalogo_ventas.raw.archivos
COMMENT 'Volume para almacenar archivos csv crudos';

SHOW VOLUMES IN catalogo_ventas.raw

In [0]:
%sql
-- PASO 4: VERIFICAR QUE SE SUBIO EL ARCHIVO CORRECTAMENTE

LIST '/Volumes/catalogo_ventas/raw/archivos'


In [0]:
%sql
-- PASO 5: CREAR LA TABLA ventas_helados_bronze

DROP TABLE IF EXISTS catalogo_ventas.raw.ventas_helados_bronze;

CREATE TABLE catalogo_ventas.raw.ventas_helados_bronze
  SELECT * FROM read_files(
    '/Volumes/catalogo_ventas/raw/archivos/ventas_helados_bronze.csv',
    format => 'csv',
    header => true,
    sep => ';'

  )


In [0]:
%sql
-- VERIFICAR TABLA CREADA

SHOW TABLES IN catalogo_ventas.raw


## **Columnas del dataset**

| Columna       | Descripcion                                       |
| ------------- | ------------------------------------------------- |
| succodigo     | Código de la sucursal                             |
| turno         | Turno de la operación (mañana, tarde, noche)      |
| caja          | identificación de la caja                         |
| venta         | código de la venta                                |
| comprobante   | Tipo o número de comprobante fiscal               |
| vtaoperacion  | Tipo de operación de venta (VF, VL, CL, VG)       |
| clinombre     | Nombre del cliente                                |
| vtaestado     | Estado de la venta (cerrada, anulada, pendiente…) |
| vtafecha      | Fecha de la venta                                 |
| usulogin      | Usuario que realizó la operación                  |
| condvtapos    | Condición de pago                                 |
| delivery      | Indica si es entrega a domicilio                  |
| articulo      | Código del artículo vendido                       |
| descrip       | Descripción del artículo                          |
| precio        | Precio unitario                                   |
| cant          | Cantidad vendida                                  |
| total         | Total de la venta                                 |
| sucursal      | Nombre o código de la sucursal                    |
| cliente       | Identificador de cliente                          |




## **EDA PASO 1: Exploracion Inicial**

In [0]:
%sql

-- TAMAÑO DEL DATASET

SELECT 
  COUNT(*) AS total_ventas
FROM catalogo_ventas.raw.ventas_helados_bronze

In [0]:
%sql
-- EXPLORAR ESTRUCTURA DEL DATASET

DESCRIBE EXTENDED catalogo_ventas.raw.ventas_helados_bronze

In [0]:
%sql

-- VER LAS 10 PRIMERAS FILAS

SELECT * FROM catalogo_ventas.raw.ventas_helados_bronze LIMIT 10


## **EDA Paso 2: Análisis de Valores Nulos**

In [0]:
%sql
-- CONTAR VALORES NULOS POR COLUMNAS

WITH total_ventas AS (

SELECT COUNT(*) AS total_ventas FROM catalogo_ventas.raw.ventas_helados_bronze

),

nulos AS (
SELECT
  COUNT(*) - COUNT(succodigo) AS nulos_sucursal,
  COUNT(*) - COUNT(turno) AS nulos_turno,
  COUNT(*) - COUNT(caja) AS nulos_caja,
  COUNT(*) - COUNT(venta) AS nulos_venta,
  COUNT(*) - COUNT(comprobante) AS nulos_comprobante,
  ROUND((count(*) - COUNT(comprobante)) * 100.0 / count(*),2) AS pct_nulos_comprobante,
  COUNT(*) - COUNT(vtaoperacion) AS nulos_vtaoperacion,
  COUNT(*) - COUNT(clinombre) AS nulos_clinombre,
  COUNT(*) - COUNT(vtaestado) AS nulos_vtaestado,
  COUNT(*) - COUNT(vtafecha) AS nulos_vtafecha,
  COUNT(*) - COUNT(usulogin) AS nulos_usulogin,
  COUNT(*) - COUNT(condvtapos) AS nulos_condvtapos,
  COUNT(*) - COUNT(delivery) AS nulos_delivery,
  COUNT(*) - COUNT(articulo) AS nulos_articulo,
  COUNT(*) - COUNT(descrip) AS nulos_descrip,
  COUNT(*) - COUNT(precio) AS nulos_precio,
  COUNT(*) - COUNT(cant) AS nulos_cant,
  COUNT(*) - COUNT(total) AS nulos_total,
  COUNT(*) - COUNT(cliente) AS nulos_cliente
FROM catalogo_ventas.raw.ventas_helados_bronze
)

SELECT
  vt.*,
  n.*
FROM total_ventas vt, nulos n

/*
Observaciones clave

Solo comprobante tiene nulos importantes (~29%).

Esto puede ser ventas en efectivo o sin facturación.

No afecta análisis de totales o productos, pero sí para análisis fiscal o de comprobantes.

Resto de columnas están completas (0%) → excelente calidad de datos para análisis de ventas, artículos, clientes y turnos.

3️⃣ Recomendaciones de tratamiento
Columna	Estrategia sugerida
comprobante	Depende del análisis:
• Si el objetivo es facturación: imputar como “No aplica” o filtrar nulos.
• Si el objetivo es ventas totales: dejar nulos, no afecta sumas.
Resto de columnas	No requieren tratamiento; listas para análisis exploratorio y agregaciones


*/
